# Bioinformatics pipeline testing - taxonomy assignment

This notebook assigns taxonomy to the sequence variants generated in the previous notebook.

Taxonomic assignment includes:
- RDP, BLAST, MapSeq and VSEARCH using a custom reference database combining MetaFishLib (fish) and MIDORI2 (non-fish).
- We also have MetaBEAT taxonomic assignment, performed externally using an older reference database.
- taxonomizr is used to condense BLAST results

**NOTE:** There are some places where the test dataset name must be changed to either `Marchamley`, `RingTrial_Sean`, `Windermere_2017` or `Simulated`. Please search and replace.

## 1. Set-up

In [0]:
%sh

# needed for packages to install
apt-get -q update

# vsearch install
apt-get -q install -y vsearch

# BLAST+ installed
apt-get -q install -y ncbi-blast+

# MAPseq
cd /dbfs/tmp # temp directory
wget https://github.com/jfmrod/MAPseq/releases/download/v2.1.1/mapseq-2.1.1-linux.tar.gz # download
tar -xvzf mapseq-2.1.1-linux.tar.gz # extract

## check install
cd /dbfs/tmp/mapseq-2.1.1-linux
./mapseq

In [0]:
# Install packages
installed <- rownames(installed.packages())

ensure_cran <- function(pkgs, repos = "https://www.stats.bris.ac.uk/R/") {
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    install.packages(to_install, dependencies = TRUE, repos = repos)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

ensure_bioc <- function(pkgs) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
  }
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    BiocManager::install(to_install, ask = FALSE, update = FALSE)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

## CRAN packages
cran_pkgs <- c("devtools", "dplyr", "tidyverse", "seqinr", "taxonomizr")
ensure_cran(cran_pkgs)

## Bioconductor packages
bioc_pkgs <- c("dada2")
ensure_bioc(bioc_pkgs)

In [0]:
test_data_name <- "Marchamley" # change here to update test data (same name as folder)
test_data_loc <- file.path("Data", "Raw", test_data_name)
results_loc = paste("Results/", test_data_name)
print(head(list.files(test_data_loc)))

## RDP (DADA2)

In [0]:
# database path
database <- file.path(getwd(), "Data", "Databases", "EA_custom_Riaz", "final_database_001.fasta")

In [0]:
source("Scripts/08a_assign_taxonomy_RDP_DADA2.R") # included MeatBEAT file processing in here too

In [0]:
source("Scripts/08c_assign_taxonomy_RDP_VSEARCH.R")

In [0]:
source("Scripts/08e_assign_taxonomy_RDP_MetaBEAT.R") # no data for simulated data, will fail when simulated run

## BLAST

In [0]:
%sh
bash Scripts/08b_assign_taxonomy_BLAST_DADA2.sh Marchamley # change to test data

In [0]:
%sh
bash Scripts/08d_assign_taxonomy_BLAST_VSEARCH.sh Marchamley # change to test data

In [0]:
%sh
bash Scripts/08e_assign_taxonomy_BLAST_MetaBEAT.sh Marchamley # change to test data, no data for simulated, will fail when simulated run

### Condense BLAST taxonomy

Condense taxonomy (lowest common ancestor) for BLAST outputs. 

In [0]:
min_pident <- 95
min_length <- 88 # 20% within 110 amplicon length
top_frac <- 0.95

In [0]:
source("Scripts/09_condensing_taxonomy_DADA2.R")

In [0]:
source("Scripts/09b_condensing_taxonomy_VSEARCH.R")

In [0]:
source("Scripts/09c_condensing_taxonomy_MetaBEAT.R") # no data for simulated, will fail when simulated run

Tidying is required for the MetaBEAT BLAST outputs.

In [0]:
%sh
python -u Scripts/09d_tidy_MetaBEAT_blast.py # parameters to change at top of script, no data for simulated

## MAPseq

In [0]:
%sh

# dataset name 
test_data_name="Marchamley" # remember to change

# input ASVs
input_fasta_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_DADA2.fasta"
input_fasta_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06b_ASV_seqs_VSEARCH.fasta"
input_fasta_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_MetaBEAT.fasta"

# MAPseq database 
db_dir="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz"

db_fasta="${db_dir}/EA_riaz_mapseq.fasta"
db_tax="${db_dir}/EA_riaz_mapseq.tax"

# output 
out_file_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_mapseq_output_dada2_EA_riaz.mseq"
out_file_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_mapseq_output_vsearch_EA_riaz.mseq"
out_file_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_mapseq_output_MetaBEAT_EA_riaz.mseq"

# dada2
/dbfs/tmp/mapseq-2.1.1-linux/mapseq \
    -nthreads 16 \
    -tophits 80 \
    -topotus 40 \
    "$input_fasta_dada2" \
    "$db_fasta" "$db_tax" \
    > "$out_file_dada2"

# vsearch
/dbfs/tmp/mapseq-2.1.1-linux/mapseq \
    -nthreads 16 \
    -tophits 80 \
    -topotus 40 \
    "$input_fasta_vsearch" \
    "$db_fasta" "$db_tax" \
    > "$out_file_vsearch"

# MetaBEAT - none for simulated
/dbfs/tmp/mapseq-2.1.1-linux/mapseq \
    -nthreads 16 \
    -tophits 80 \
    -topotus 40 \
    "$input_fasta_MetaBEAT" \
    "$db_fasta" "$db_tax" \
    > "$out_file_MetaBEAT"

In [0]:
%sh
python -u Scripts/09e_tidy_mapseq.py # parameters to change at top of script

## VSEARCH

This will run usearch_global for taxonomy assignment and then EBI's custom LCA script to condense taxonomy.

In [0]:
%sh

# dataset name 
test_data_name="Marchamley" # remember to change

# input ASVs
input_fasta_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_DADA2.fasta"
input_fasta_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06b_ASV_seqs_VSEARCH.fasta"
input_fasta_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_MetaBEAT.fasta"

# output files
out_file_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_vsearch_output_dada2_EA_riaz_lca.tsv"
out_file_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_vsearch_output_vsearch_EA_riaz_lca.tsv"
out_file_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_vsearch_output_MetaBEAT_EA_riaz_lca.tsv"

# vsearch database 
db_dir="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/vsearch_indexed_reference"

#dada2
vsearch --usearch_global "$input_fasta_dada2" --db "$db_dir" --threads 4 --userout /local_disk0/tmp.tsv --strand both --id 0.50 --maxaccepts 0 --maxrejects 0 --query_cov 0.9 --userfields query+target+id+qcov 
outfile="${file%.tsv}_lca.tsv"
python Scripts/09e_lca_pid_qcov_filtering_top.py "/local_disk0/tmp.tsv" > "$out_file_dada2"

#vsearch
vsearch --usearch_global "$input_fasta_vsearch" --db "$db_dir" --threads 4 --userout /local_disk0/tmp.tsv --strand both --id 0.50 --maxaccepts 0 --maxrejects 0 --query_cov 0.9 --userfields query+target+id+qcov 
outfile="${file%.tsv}_lca.tsv"
python Scripts/09e_lca_pid_qcov_filtering_top.py "/local_disk0/tmp.tsv" > "$out_file_vsearch"

#metabeat - none for simulated
vsearch --usearch_global "$input_fasta_MetaBEAT" --db "$db_dir" --threads 4 --userout /local_disk0/tmp.tsv --strand both --id 0.50 --maxaccepts 0 --maxrejects 0 --query_cov 0.9 --userfields query+target+id+qcov 
outfile="${file%.tsv}_lca.tsv"
python Scripts/09e_lca_pid_qcov_filtering_top.py "/local_disk0/tmp.tsv" > "$out_file_MetaBEAT"

# checks

echo "DADA2:"
wc -l "$out_file_dada2"
ls -lh "$out_file_dada2"

echo "VSEARCH:"
wc -l "$out_file_vsearch"
ls -lh "$out_file_vsearch"

echo "MetaBEAT:"
wc -l "$out_file_MetaBEAT"
ls -lh "$out_file_MetaBEAT"

In [0]:
%skip
%sh 

#MetaBEAT kept many more sequences than dada2 and vsearch meaning the output file for vsearch couldn't be stored with all hits. Data after LCA was small enough so that was saved within this dir.
# none for simulated data 

test_data_name="RingTrial_Sean" # remember to change

input_fasta_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_MetaBEAT.fasta"
out_file_MetaBEAT_lca="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_vsearch_output_MetaBEAT_EA_riaz_lca.tsv"

db_dir="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom_Riaz/vsearch_indexed_reference"

#metabeat - none for simulated
vsearch --usearch_global "$input_fasta_MetaBEAT" --db "$db_dir" --threads 4 --userout /local_disk0/tmp.tsv --strand both --id 0.50 --maxaccepts 0 --maxrejects 0 --query_cov 0.9 --userfields query+target+id+qcov 
outfile="${file%.tsv}_lca.tsv"
python Scripts/09e_lca_pid_qcov_filtering_top.py "/local_disk0/tmp.tsv" > "$out_file_MetaBEAT_lca"

In [0]:
%skip
%sh

for dataset in Marchamley RingTrial_Sean Windermere_2017 Simulated
do
    for file in Data/Processed/"$dataset"/08_vsearch_output*.tsv
    do
        [[ "$file" == *_lca.tsv ]] && continue
        [[ "$file" == *MeatBEAT* ]] && continue

        outfile="${file%.tsv}_lca.tsv"

        python Scripts/09e_lca_pid_qcov_filtering_top.py "$file" > "$outfile"
    done
done

End of notebook. Please continue to the next notebook to explore comparisons.